Aplikasi Streamlit — Klasifikasi Penyakit Daun Tomat (Aplikasi Perbandingan)

## 1. Struktur Folder

In [1]:
import os
import shutil
from pathlib import Path

OUTPUT_DIR = str(Path.home() / "Documents" / "Skripsi" / "SVM_bertahap")
SKRIPSI_DIR = Path.home() / "Documents" / "Skripsi"
APP_DIR = str(SKRIPSI_DIR / "Aplikasi_perbandingan")   # folder aplikasi baru

for sub in ["utils", "controllers", "views"]:
    os.makedirs(os.path.join(APP_DIR, sub), exist_ok=True)
    open(os.path.join(APP_DIR, sub, "__init__.py"), "a").close()

src = os.path.join(OUTPUT_DIR, "feature_utils.py")
dst = os.path.join(APP_DIR, "utils", "feature_utils.py")
shutil.copy(src, dst)
print(f"feature_utils.py -> {dst}")

feature_utils.py -> C:\Users\lenovo\Documents\Skripsi\Aplikasi_perbandingan\utils\feature_utils.py


## 2. Model Loader

In [2]:
model_loader_code = '''
import os
import joblib
import streamlit as st
from pathlib import Path

ARTIFACT_DIR = str(Path.home() / "Documents" / "Skripsi" / "SVM_bertahap")

REQUIRED_ARTIFACTS = [
    "svm_m1_raw.pkl", "svm_m2_raw_glcm.pkl", "svm_m3_raw_glcm_hog_lbp.pkl",
    "raw_scaler.pkl", "glcm_scaler.pkl", "hog_scaler.pkl", "lbp_scaler.pkl",
    "feature_config.json",
]

def artifact_path(filename):
    return os.path.join(ARTIFACT_DIR, filename)

def check_artifacts():
    return {f: os.path.exists(artifact_path(f)) for f in REQUIRED_ARTIFACTS}

@st.cache_resource(show_spinner="Memuat model M1...")
def load_m1():
    return joblib.load(artifact_path("svm_m1_raw.pkl"))

@st.cache_resource(show_spinner="Memuat model M2...")
def load_m2():
    return joblib.load(artifact_path("svm_m2_raw_glcm.pkl"))

@st.cache_resource(show_spinner="Memuat model M3...")
def load_m3():
    return joblib.load(artifact_path("svm_m3_raw_glcm_hog_lbp.pkl"))

@st.cache_resource(show_spinner="Memuat scaler fitur...")
def load_scalers():
    return {
        "raw": joblib.load(artifact_path("raw_scaler.pkl")),
        "glcm": joblib.load(artifact_path("glcm_scaler.pkl")),
        "hog": joblib.load(artifact_path("hog_scaler.pkl")),
        "lbp": joblib.load(artifact_path("lbp_scaler.pkl")),
    }

def load_all_models():
    return {
        "models": {"M1": load_m1(), "M2": load_m2(), "M3": load_m3()},
        "scalers": load_scalers(),
    }
'''

with open(os.path.join(APP_DIR, "utils", "model_loader.py"), "w", encoding="utf-8") as f:
    f.write(model_loader_code)

print("model_loader.py tersimpan")

model_loader.py tersimpan


## 3. Data Loader

In [3]:
data_loader_code = '''
import os
import json
import streamlit as st

from utils.model_loader import artifact_path

_DEFAULT_CONFIG = {
    "img_size": 128,
    "raw_pixel_size": 32,
    "classes": [
        "Tomato___Early_blight", "Tomato___healthy",
        "Tomato___Late_blight", "Tomato___Septoria_leaf_spot",
    ],
    "class_names": ["Early Blight", "Healthy", "Late Blight", "Septoria"],
}

@st.cache_data(show_spinner="Memuat konfigurasi fitur...")
def load_feature_config():
    path = artifact_path("feature_config.json")
    if not os.path.exists(path):
        return _DEFAULT_CONFIG
    with open(path) as f:
        return json.load(f)
'''

with open(os.path.join(APP_DIR, "utils", "data_loader.py"), "w", encoding="utf-8") as f:
    f.write(data_loader_code)

print("data_loader.py tersimpan")

data_loader.py tersimpan


## 4. Controller Klasifikasi

In [4]:
klasifikasi_controller_code = '''
import numpy as np

from utils.feature_utils import (
    preprocess_image, extract_raw_pixel_features,
    extract_glcm_features, extract_hog_features, extract_lbp_features,
)

CONFIDENCE_THRESHOLD = 0.60

def extract_feature_vector(img, model_choice, scalers, config):
    raw_feat = scalers["raw"].transform(
        extract_raw_pixel_features(img, config["raw_pixel_size"])
    )
    feature_vector = raw_feat

    if model_choice in ("M2", "M3"):
        glcm_feat = scalers["glcm"].transform(
            extract_glcm_features(img, config["glcm_distances"],
                                   config["glcm_angles"], config["glcm_props"])
        )
        feature_vector = np.concatenate([feature_vector, glcm_feat], axis=1)

    if model_choice == "M3":
        hog_feat = scalers["hog"].transform(
            extract_hog_features(img, config["hog_orientations"],
                                  config["hog_pixels_per_cell"], config["hog_cells_per_block"])
        )
        lbp_feat = scalers["lbp"].transform(
            extract_lbp_features(img, config["lbp_p"], config["lbp_r"])
        )
        feature_vector = np.concatenate([feature_vector, hog_feat, lbp_feat], axis=1)

    return feature_vector

def predict_image(image_bgr, model_choice, resources, config, class_names):
    if model_choice not in ("M1", "M2", "M3"):
        raise ValueError(f"Model tidak dikenal: {model_choice}")

    img = preprocess_image(image_bgr, config["img_size"])
    scalers = resources["scalers"]
    model = resources["models"][model_choice]

    feature_vector = extract_feature_vector(img, model_choice, scalers, config)

    proba = model.predict_proba(feature_vector)[0]
    max_proba = float(np.max(proba))
    pred_idx = int(np.argmax(proba))
    is_confident = max_proba >= CONFIDENCE_THRESHOLD

    return {
        "model": model_choice,
        "pred_idx": pred_idx,
        "pred_label": class_names[pred_idx] if is_confident else None,
        "is_confident": is_confident,
        "max_proba": max_proba,
        "all_proba": proba,
    }

def predict_all_models(image_bgr, resources, config, class_names):
    results = {}
    for model_choice in ("M1", "M2", "M3"):
        results[model_choice] = predict_image(image_bgr, model_choice, resources, config, class_names)
    return results
'''

with open(os.path.join(APP_DIR, "controllers", "klasifikasi_controller.py"), "w", encoding="utf-8") as f:
    f.write(klasifikasi_controller_code)

print("klasifikasi_controller.py tersimpan")

klasifikasi_controller.py tersimpan


## 5. Controller Pemrosesan 1–50 Citra (baru)

In [5]:
batch_controller_code = r'''
import cv2
import numpy as np
import pandas as pd

from controllers.klasifikasi_controller import predict_all_models, CONFIDENCE_THRESHOLD

MAX_CITRA = 50
MODEL_KEYS = ("M1", "M2", "M3")
LABEL_TIDAK_DIKENALI = "Tidak dikenali"


def decode_image(image_bytes):
    if not image_bytes:
        return None   # berkas kosong (0 byte) diperlakukan sebagai citra gagal dibaca
    file_bytes = np.frombuffer(image_bytes, dtype=np.uint8)
    return cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)


def make_unique_names(names):
    seen = {}
    unique = []
    for name in names:
        if name not in seen:
            seen[name] = 1
            unique.append(name)
            continue
        seen[name] += 1
        stem, dot, ext = name.rpartition(".")
        unique.append(f"{stem} ({seen[name]}).{ext}" if dot else f"{name} ({seen[name]})")
    return unique


def predict_batch(items, resources, config, class_names, progress_callback=None):
    if len(items) > MAX_CITRA:
        raise ValueError(f"Jumlah citra {len(items)} melebihi batas {MAX_CITRA}")

    batch_results = []
    total = len(items)
    for idx, item in enumerate(items):
        image_bgr = decode_image(item["bytes"])
        if image_bgr is None:
            batch_results.append({"index": idx, "name": item["name"],
                                  "error": "Citra gagal dibaca", "result": None})
        else:
            result = predict_all_models(image_bgr, resources, config, class_names)
            batch_results.append({"index": idx, "name": item["name"],
                                  "error": None, "result": result})
        if progress_callback is not None:
            progress_callback(idx + 1, total, item["name"])
    return batch_results


def agreement_status(comparison_result):
    labels = [comparison_result[m]["pred_label"] for m in MODEL_KEYS]
    if all(l is not None for l in labels) and len(set(labels)) == 1:
        return "Sepakat"
    if any(l is None for l in labels):
        return "Ada yang tidak dikenali"
    return "Tidak sepakat"


def display_label(model_result):
    return model_result["pred_label"] if model_result["is_confident"] else LABEL_TIDAK_DIKENALI


def build_result_table(batch_results):
    rows = []
    for item in batch_results:
        row = {"No": item["index"] + 1, "Nama berkas": item["name"]}
        if item["error"] is not None:
            for m in MODEL_KEYS:
                row[f"Prediksi {m}"] = "-"
                row[f"Keyakinan {m} (%)"] = float("nan")
            row["Status"] = item["error"]
        else:
            res = item["result"]
            for m in MODEL_KEYS:
                row[f"Prediksi {m}"] = display_label(res[m])
                row[f"Keyakinan {m} (%)"] = round(res[m]["max_proba"] * 100, 2)
            row["Status"] = agreement_status(res)
        rows.append(row)
    return pd.DataFrame(rows)


def build_export_table(batch_results, class_names):
    rows = []
    for item in batch_results:
        row = {"no": item["index"] + 1, "nama_berkas": item["name"]}
        res = item["result"]
        for m in MODEL_KEYS:
            if res is None:
                row[f"pred_{m}"] = ""
                row[f"yakin_{m}"] = None
                row[f"argmax_{m}"] = ""
                for c in class_names:
                    row[f"p_{m}_{c.replace(' ', '')}"] = None
                continue
            r = res[m]
            row[f"pred_{m}"] = display_label(r)
            row[f"yakin_{m}"] = round(r["max_proba"] * 100, 2)
            row[f"argmax_{m}"] = class_names[r["pred_idx"]]
            for j, c in enumerate(class_names):
                row[f"p_{m}_{c.replace(' ', '')}"] = round(float(r["all_proba"][j]) * 100, 2)
        row["status"] = agreement_status(res) if res is not None else ""
        row["galat"] = item["error"] or ""
        row["ambang_keyakinan"] = CONFIDENCE_THRESHOLD
        rows.append(row)
    return pd.DataFrame(rows)


def prediction_distribution(batch_results, class_names):
    order = list(class_names) + [LABEL_TIDAK_DIKENALI]
    counts = {m: {lbl: 0 for lbl in order} for m in MODEL_KEYS}
    for item in batch_results:
        if item["result"] is None:
            continue
        for m in MODEL_KEYS:
            counts[m][display_label(item["result"][m])] += 1
    return pd.DataFrame(counts).loc[order]
'''

with open(os.path.join(APP_DIR, "controllers", "batch_controller.py"), "w", encoding="utf-8") as f:
    f.write(batch_controller_code)

print("batch_controller.py tersimpan")

batch_controller.py tersimpan


## 6. View Klasifikasi (satu alur untuk 1–50 citra)

Tiga langkah:
1. **Unggah Citra** — satu *uploader* yang menerima 1 sampai 50 berkas; tombol lanjut terkunci bila melebihi 50.
2. **Pratinjau** — satu citra ditampilkan besar beserta nama dan ukuran berkas; banyak citra ditampilkan sebagai galeri.
3. **Hasil Klasifikasi** — satu citra langsung menampilkan perbandingan M1, M2, M3; banyak citra menampilkan ringkasan,
   tabel hasil, tombol unduh CSV, lalu perbandingan rinci untuk citra yang dipilih.
   Perbandingan disusun menyamping dalam tiga kolom: citra dan pesan kesepakatan, hasil ketiga model, serta grafik
   probabilitas, sehingga seluruh perbandingan tampil dalam satu layar.

In [6]:
klasifikasi_view_code = r'''
import time
import pandas as pd
import streamlit as st

from controllers.klasifikasi_controller import CONFIDENCE_THRESHOLD
from controllers.batch_controller import (
    MAX_CITRA, MODEL_KEYS, predict_batch, agreement_status, make_unique_names,
    build_result_table, build_export_table,
)

MODEL_LABELS = {
    "M1": "M1 - SVM + Raw Pixel",
    "M2": "M2 - SVM + Raw Pixel + GLCM",
    "M3": "M3 - SVM + Raw Pixel + GLCM + HOG + LBP",
}

STEPS = [
    ("input", "1. Unggah Citra"),
    ("preview", "2. Pratinjau"),
    ("hasil", "3. Hasil Klasifikasi"),
]

GRID_COLS = 5
THUMB_WIDTH = 150


def _init_state():
    if "page" not in st.session_state:
        st.session_state.page = "input"
    if "uploader_key" not in st.session_state:
        st.session_state.uploader_key = 0


def _go_to(page_name):
    st.session_state.page = page_name


def _reset_all():
    st.session_state.page = "input"
    st.session_state.uploader_key += 1
    for key in ("items", "batch_results", "elapsed"):
        st.session_state.pop(key, None)


def _safe_image(image_bytes, caption=None, width=None):
    try:
        st.image(image_bytes, caption=caption, width=width)
    except Exception:
        st.warning("Citra tidak dapat ditampilkan (berkas rusak atau bukan citra).")
        if caption:
            st.caption(caption)


def _short(name, limit=26):
    return name if len(name) <= limit else name[:limit - 3] + "..."


def _render_step_indicator(current_page):
    cols = st.columns(len(STEPS))
    for col, (key, label) in zip(cols, STEPS):
        with col:
            if key == current_page:
                st.markdown(f"**:blue[{label}]**")
            else:
                st.markdown(f":gray[{label}]")
    st.divider()


def render(resources, config, class_names):
    _init_state()

    st.header("Klasifikasi Penyakit Daun Tomat - SVM")
    st.caption("Fusi fitur bertahap: Raw Pixel (M1) -> + GLCM (M2) -> + GLCM + HOG + LBP (M3)")
    _render_step_indicator(st.session_state.page)

    if st.session_state.page == "input":
        _render_input_step()
    elif st.session_state.page == "preview":
        _render_preview_step(resources, config, class_names)
    else:
        _render_hasil_step(class_names)


# ------------------------------------------------------------------ Langkah 1
def _render_input_step():
    with st.container(border=True):
        st.subheader("Langkah 1 - Unggah Citra Daun Tomat")
        st.write(
            f"Unggah satu hingga {MAX_CITRA} citra daun tomat (format JPG atau PNG). Setiap citra "
            "akan diklasifikasikan sekaligus oleh ketiga model (M1, M2, M3) untuk dibandingkan "
            "hasilnya secara langsung."
        )
        uploaded_files = st.file_uploader(
            "Pilih citra", type=["jpg", "jpeg", "png"], accept_multiple_files=True,
            key=f"uploader_{st.session_state.uploader_key}",
        )
        n_files = len(uploaded_files) if uploaded_files else 0
        too_many = n_files > MAX_CITRA

        if too_many:
            st.error(
                f"Terpilih {n_files} citra, melebihi batas {MAX_CITRA} citra. "
                f"Hapus {n_files - MAX_CITRA} citra dari daftar unggahan terlebih dahulu."
            )
        elif n_files:
            st.caption(f"{n_files} dari maksimal {MAX_CITRA} citra terpilih.")

        st.write("")
        next_clicked = st.button(
            "Lanjut ke Pratinjau ->", type="primary", disabled=(n_files == 0 or too_many),
        )

    if next_clicked and n_files and not too_many:
        names = make_unique_names([f.name for f in uploaded_files])
        st.session_state.items = [
            {"name": name, "bytes": f.getvalue()} for name, f in zip(names, uploaded_files)
        ]
        st.session_state.pop("batch_results", None)
        _go_to("preview")
        st.rerun()


# ------------------------------------------------------------------ Langkah 2
def _render_preview_step(resources, config, class_names):
    items = st.session_state.get("items")
    if not items:
        st.warning("Belum ada citra yang diunggah.")
        _go_to("input")
        st.rerun()
        return

    with st.container(border=True):
        st.subheader("Langkah 2 - Pratinjau Citra")

        if len(items) == 1:
            item = items[0]
            col_img, col_info = st.columns([2, 1])
            with col_img:
                _safe_image(item["bytes"], caption=item["name"], width=320)
            with col_info:
                st.markdown("**Nama berkas**")
                st.write(item["name"])
                st.markdown("**Ukuran berkas**")
                st.write(f"{len(item['bytes']) / 1024:.1f} KB")
                st.markdown("**Model yang akan dijalankan**")
                for label in MODEL_LABELS.values():
                    st.write(f"- {label}")
        else:
            total_kb = sum(len(item["bytes"]) for item in items) / 1024
            col_a, col_b, col_c = st.columns(3)
            col_a.metric("Jumlah citra", f"{len(items)} / {MAX_CITRA}")
            col_b.metric("Total ukuran", f"{total_kb:.1f} KB")
            col_c.metric("Model yang dijalankan", "M1, M2, M3")
            for start in range(0, len(items), GRID_COLS):
                cols = st.columns(GRID_COLS)
                for offset, item in enumerate(items[start:start + GRID_COLS]):
                    with cols[offset]:
                        _safe_image(item["bytes"], width=THUMB_WIDTH)
                        st.caption(f"{start + offset + 1}. {_short(item['name'])}  \n"
                                   f"{len(item['bytes']) / 1024:.1f} KB")

        st.write("")
        col_back, col_run = st.columns(2)
        with col_back:
            back_clicked = st.button("<- Ganti Citra", use_container_width=True)
        with col_run:
            run_label = "Jalankan Klasifikasi" if len(items) == 1 else f"Jalankan Klasifikasi ({len(items)} citra)"
            run_clicked = st.button(run_label, type="primary", use_container_width=True)

    if back_clicked:
        _reset_all()
        st.rerun()

    if run_clicked:
        progress = st.progress(0.0, text="Menjalankan prediksi M1, M2, dan M3...")

        def _update(done, total, name):
            progress.progress(done / total, text=f"Memproses citra {done}/{total}: {_short(name, 40)}")

        start_time = time.perf_counter()
        batch_results = predict_batch(items, resources, config, class_names, _update)
        st.session_state.elapsed = time.perf_counter() - start_time

        if all(r["result"] is None for r in batch_results):
            st.error("Citra gagal dibaca. Silakan kembali dan unggah ulang berkas JPG/PNG lain.")
            return

        st.session_state.batch_results = batch_results
        _go_to("hasil")
        st.rerun()


# ------------------------------------------------------------------ Langkah 3
def _render_agreement(comparison_result):
    labels = [comparison_result[m]["pred_label"] for m in MODEL_KEYS]
    all_agree = (
        len(set(l for l in labels if l is not None)) == 1
        and all(l is not None for l in labels)
    )
    if all_agree:
        st.success(f"Ketiga model sepakat memprediksi: **{labels[0]}**")
    else:
        st.info("Ketiga model tidak sepenuhnya sepakat. Lihat rincian tiap model di samping.")


def _render_model_cards(comparison_result):
    for model_choice in MODEL_KEYS:
        result = comparison_result[model_choice]
        with st.container(border=True):
            st.markdown(f"**{MODEL_LABELS[model_choice]}**")
            if result["is_confident"]:
                delta_text = f'{result["max_proba"]*100:.1f}% yakin'
                st.metric("Prediksi", result["pred_label"], delta_text)
            else:
                threshold_pct = f"{CONFIDENCE_THRESHOLD*100:.0f}%"
                delta_text = f'{result["max_proba"]*100:.1f}% (di bawah ambang {threshold_pct})'
                st.metric("Prediksi", "Tidak dikenali", delta_text)


def _render_probability(comparison_result, class_names):
    st.markdown("**Grafik Probabilitas per Kelas**")
    proba_compare_df = pd.DataFrame({
        model_choice: comparison_result[model_choice]["all_proba"]
        for model_choice in MODEL_KEYS
    })
    proba_compare_df.index = list(class_names)
    st.bar_chart(proba_compare_df, height=320)
    with st.expander("Lihat detail probabilitas dalam bentuk tabel"):
        st.dataframe(proba_compare_df.style.format("{:.2%}"), use_container_width=True)


def _render_hasil_step(class_names):
    batch_results = st.session_state.get("batch_results")
    items = st.session_state.get("items", [])
    if batch_results is None:
        st.warning("Belum ada hasil klasifikasi.")
        _go_to("input")
        st.rerun()
        return

    ok = [r for r in batch_results if r["result"] is not None]
    failed = [r["name"] for r in batch_results if r["result"] is None]

    # Ringkasan dan tabel hanya ditampilkan bila citra lebih dari satu.
    if len(batch_results) > 1:
        n_agree = sum(agreement_status(r["result"]) == "Sepakat" for r in ok)
        with st.container(border=True):
            st.subheader("Langkah 3 - Ringkasan Hasil M1, M2, dan M3")
            col_a, col_b, col_c, col_d = st.columns(4)
            col_a.metric("Citra diproses", f"{len(ok)} / {len(batch_results)}")
            col_b.metric("Ketiga model sepakat", f"{n_agree} citra")
            col_c.metric("Belum sepakat", f"{len(ok) - n_agree} citra")
            col_d.metric("Waktu proses", f"{st.session_state.get('elapsed', 0.0):.1f} detik")
            if failed:
                st.error("Citra berikut gagal dibaca dan dilewati: " + ", ".join(failed))

            table = build_result_table(batch_results)
            fmt = {f"Keyakinan {m} (%)": "{:.2f}" for m in MODEL_KEYS}
            st.dataframe(table.style.format(fmt, na_rep="-"), use_container_width=True, hide_index=True)
            st.download_button(
                "Unduh hasil (CSV)",
                data=build_export_table(batch_results, class_names).to_csv(index=False).encode("utf-8"),
                file_name="hasil_aplikasi_batch.csv", mime="text/csv",
            )
        st.write("")

    # Perbandingan disusun menyamping dalam tiga kolom: citra, hasil ketiga model, dan grafik probabilitas.
    with st.container(border=True):
        if len(batch_results) == 1:
            st.subheader("Langkah 3 - Perbandingan Hasil M1, M2, dan M3")
        else:
            st.subheader("Perbandingan Hasil per Citra")
        col_img, col_models, col_chart = st.columns([1, 1, 1.2], gap="medium")
        with col_img:
            if len(batch_results) == 1:
                chosen = ok[0]
            else:
                pos = st.selectbox(
                    "Pilih citra", options=list(range(len(ok))),
                    format_func=lambda i: f"{ok[i]['index'] + 1}. {ok[i]['name']}", key="detail_pos",
                )
                chosen = ok[pos]
            st.image(items[chosen["index"]]["bytes"], caption=chosen["name"], width=280)
            _render_agreement(chosen["result"])
        with col_models:
            _render_model_cards(chosen["result"])
        with col_chart:
            _render_probability(chosen["result"], class_names)

    st.write("")
    st.button("Klasifikasikan Citra Baru", type="primary", on_click=_reset_all)
'''

with open(os.path.join(APP_DIR, "views", "klasifikasi_view.py"), "w", encoding="utf-8") as f:
    f.write(klasifikasi_view_code)

stale = os.path.join(APP_DIR, "views", "batch_view.py")
if os.path.exists(stale):
    os.remove(stale)   # sisa versi sebelumnya yang memakai pilihan mode

print("klasifikasi_view.py tersimpan")

klasifikasi_view.py tersimpan


## 7. Entry Point (main.py)

In [7]:
main_code = '''
import sys
import os

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import streamlit as st

from utils.model_loader import load_all_models, check_artifacts
from utils.data_loader import load_feature_config
from controllers.batch_controller import MAX_CITRA
from views import klasifikasi_view

st.set_page_config(page_title="Klasifikasi Penyakit Daun Tomat", page_icon="tomato", layout="wide")
st.markdown(
    "<style>.block-container, [data-testid='stMainBlockContainer'] "
    "{max-width: 1180px; padding-left: 2rem; padding-right: 2rem;}</style>",
    unsafe_allow_html=True,
)

st.sidebar.title("Klasifikasi Penyakit Daun Tomat")
st.sidebar.caption("Fusi fitur bertahap SVM: Raw Pixel vs +GLCM vs +GLCM+HOG+LBP")
st.sidebar.divider()
st.sidebar.markdown(
    f"Aplikasi ini mengunggah **satu hingga {MAX_CITRA} citra** daun tomat, lalu membandingkan "
    "hasil prediksi ketiga model (M1, M2, M3) atas setiap citra secara langsung."
)

missing = [name for name, ok in check_artifacts().items() if not ok]
if missing:
    st.error(
        "Artefak berikut belum ditemukan di direktori hasil eksperimen:\\n\\n"
        + "\\n".join(f"- {m}" for m in missing)
        + "\\n\\nJalankan notebook eksperimen sampai selesai, lalu muat ulang aplikasi ini."
    )
    st.stop()

try:
    config = load_feature_config()
    CLASS_NAMES = config.get("class_names", ["Early Blight", "Healthy", "Late Blight", "Septoria"])
    resources = load_all_models()
except Exception as e:
    st.error(f"Gagal memuat artefak: {e}")
    st.stop()

klasifikasi_view.render(resources, config, CLASS_NAMES)
'''

with open(os.path.join(APP_DIR, "main.py"), "w", encoding="utf-8") as f:
    f.write(main_code)

print("main.py tersimpan")

main.py tersimpan


## 8. Menjalankan Aplikasi

In [8]:
import socket
import subprocess
import time
import webbrowser


def port_kosong(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("localhost", port)) != 0


PORT = next(p for p in range(8502, 8600) if port_kosong(p))

subprocess.Popen(
    ["streamlit", "run", "main.py",
     "--server.port", str(PORT), "--server.headless", "true"],
    cwd=APP_DIR,
)

time.sleep(5)
webbrowser.open(f"http://localhost:{PORT}")
print("Folder aplikasi :", APP_DIR)
print(f"Aplikasi berjalan di http://localhost:{PORT}")
print("Port 8501 masih dipakai aplikasi lama:", not port_kosong(8501))

Folder aplikasi : C:\Users\lenovo\Documents\Skripsi\Aplikasi_perbandingan
Aplikasi berjalan di http://localhost:8502
Port 8501 masih dipakai aplikasi lama: False
